# Feature Engineering Using Scikit-Learn

### Do You Want to Build a Snowman?

Let's prep some data for a model to predict the heigth of snowman!

<img src = "./assets/olaf.jpeg">

#### Load Packages

In [1]:
# data analysis stack
import numpy as np
import pandas as pd

# data visualization stack
import matplotlib.pyplot as plt

%matplotlib inline
import seaborn as sns

sns.set_style("whitegrid")

# machine-learning stack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    KBinsDiscretizer,
    PolynomialFeatures,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# miscellaneous
import warnings

warnings.filterwarnings("ignore")

#### Load Data

In [2]:
data = {
    "temp": [-3, 5, 0, 7, 3, -1, 1, None, -6, 3, 0, -1, None, -2],
    "lunch": [
        "soup",
        "sandwich",
        "soup",
        "burger",
        "sandwich",
        "soup",
        "cereal",
        "salad",
        "sandwich",
        "burger",
        "soup",
        "cereal",
        "burger",
        "soup",
    ],
    "dinner": [
        "pizza",
        "pizza",
        "noodles",
        None,
        "fishsticks",
        "pizza",
        None,
        "fishsticks",
        "noodles",
        "pizza",
        None,
        "pizza",
        "fishsticks",
        "pizza",
    ],
    "precipitation": [
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "no",
        "yes",
        "yes",
        "yes",
        "no",
    ],
    "height_snowman_cm": [100, 0, 75, 0, 20, 25, 0, 35, 170, 0, 85, 85, 45, 0],
}

df_train = pd.DataFrame(data=data)
df_train

,temp,lunch,dinner,precipitation,height_snowman_cm
0,-3.0,soup,pizza,yes,100
1,5.0,sandwich,pizza,no,0
2,0.0,soup,noodles,yes,75
3,7.0,burger,NaN,yes,0
4,3.0,sandwich,fishsticks,yes,20
5,-1.0,soup,pizza,yes,25
6,1.0,cereal,NaN,no,0
7,NaN,salad,fishsticks,yes,35
8,-6.0,sandwich,noodles,yes,170
9,3.0,burger,pizza,no,0


#### Exercise
Transform the data above using scikit-learn tools in a way that is suitable for modeling
+ **Separate the DataFrame `df_train` into `X_train` and `y_train`** 
   +  Our target variable is `height_snowman_cm`
+ **Preprocess `X_train`**:
  + Identify which variables are **binary**, **categorical** and  **numeric**
  + Check which variables have **missing values**
    + **Impute missing values** as needed using appropriate strategies
  + Determine if categorical variables have **non-numeric values**
    + **Encode categorical variables** using techniques such as one-hot encoding
  + Determine if numeric variables are on different scales
    + **Scale numeric variables**
+ **Create `X_train_fe`**:
    + Once the preprocessing steps are completed, compile the transformed columns into a new DataFrame called `X_train_fe`. 


#### 1. Separate `df_train` into `X_train` and `y_train`

In [3]:
target = "height_snowman_cm"

X_train = df_train.drop(columns=target)
y_train = df_train[target]
X_train

,temp,lunch,dinner,precipitation
0,-3.0,soup,pizza,yes
1,5.0,sandwich,pizza,no
2,0.0,soup,noodles,yes
3,7.0,burger,NaN,yes
4,3.0,sandwich,fishsticks,yes
5,-1.0,soup,pizza,yes
6,1.0,cereal,NaN,no
7,NaN,salad,fishsticks,yes
8,-6.0,sandwich,noodles,yes
9,3.0,burger,pizza,no


#### 2. Identify binary, categorical and numeric variables

In [4]:
X_train.dtypes

temp             float64
lunch                str
dinner               str
precipitation        str
dtype: object

| Column | Type |
|---|---|
| `temp` | numeric |
| `lunch` | categorical (nominal) |
| `dinner` | categorical (nominal) |
| `precipitation` | categorical (binary) |

#### 3. Check for missing values

In [5]:
X_train.isna().sum()

temp             2
lunch            0
dinner           3
precipitation    0
dtype: int64

`temp` and `dinner` contain missing values and need to be imputed.

#### 4. Impute missing values

In [6]:
numeric_features = ["temp"]
categorical_features = ["lunch", "dinner", "precipitation"]

num_imputer = SimpleImputer(strategy="mean").set_output(transform="pandas")
X_train_num_imputed = num_imputer.fit_transform(X_train[numeric_features])
X_train_num_imputed

,temp
0,-3.0
1,5.0
2,0.0
3,7.0
4,3.0
5,-1.0
6,1.0
7,0.5
8,-6.0
9,3.0


In [7]:
cat_imputer = SimpleImputer(strategy="most_frequent").set_output(
    transform="pandas"
)
X_train_cat_imputed = cat_imputer.fit_transform(X_train[categorical_features])
X_train_cat_imputed

,lunch,dinner,precipitation
0,soup,pizza,yes
1,sandwich,pizza,no
2,soup,noodles,yes
3,burger,pizza,yes
4,sandwich,fishsticks,yes
5,soup,pizza,yes
6,cereal,pizza,no
7,salad,fishsticks,yes
8,sandwich,noodles,yes
9,burger,pizza,no


#### 5. Encode categorical variables
All three categorical columns hold non-numeric string values, so they need encoding. `precipitation` is binary, the other two are nominal, so one-hot encoding works for all of them (dropping the redundant column for the binary case).

In [8]:
ohe_encoder = OneHotEncoder(sparse_output=False, drop="if_binary").set_output(
    transform="pandas"
)
X_train_cat_encoded = ohe_encoder.fit_transform(X_train_cat_imputed)
X_train_cat_encoded

,lunch_burger,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,dinner_fishsticks,dinner_noodles,dinner_pizza,precipitation_yes
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
5,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
6,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
8,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
9,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


#### 6. Scale numeric variables
`temp` is the only numeric feature; scaling it puts it on a comparable range to the encoded 0/1 columns.

In [9]:
scaler = StandardScaler().set_output(transform="pandas")
X_train_num_scaled = scaler.fit_transform(X_train_num_imputed)
X_train_num_scaled

,temp
0,-1.102865
1,1.417970
2,-0.157552
3,2.048179
4,0.787761
5,-0.472657
6,0.157552
7,0.000000
8,-2.048179
9,0.787761


#### 7. Create `X_train_fe`

In [10]:
X_train_fe = pd.concat([X_train_num_scaled, X_train_cat_encoded], axis=1)
X_train_fe

,temp,lunch_burger,lunch_cereal,lunch_salad,lunch_sandwich,lunch_soup,dinner_fishsticks,dinner_noodles,dinner_pizza,precipitation_yes
0,-1.102865,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
1,1.417970,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,-0.157552,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,2.048179,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
4,0.787761,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
5,-0.472657,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
6,0.157552,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,0.000000,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
8,-2.048179,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
9,0.787761,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
